In [5]:
# CS2050: Pattern Recognition and Machine Learning
# Course Project: Hand-drawn Sketch Recognition Demo

# This notebook allows you to:
# 1. Mount Google Drive and extract a sketch dataset.
# 2. Train a model using CNN features (ResNet50) and KNN classification.
# 3. Draw a sketch interactively in Colab and get predictions.

# ## Step 1: Setup and Imports

# Install necessary libraries (usually pre-installed in Colab, but ensuring availability)
!pip install -q tensorflow scikit-learn opencv-python matplotlib tqdm

# Import libraries
import os
import cv2
import numpy as np
from google.colab import drive, output
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
import base64

# ## Step 2: Mount Google Drive and Extract Dataset

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Define paths
zip_path = "/content/drive/My Drive/Dataset_PNG.zip"
extract_path = "/content/Dataset_PNG"

# Create extract folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Extract the zip file
import zipfile
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Set dataset path
dataset_path = "/content/Dataset_PNG/sketches_png/png"
print("Categories available:", os.listdir(dataset_path))

# ## Step 3: Load Data and Train the Model

# Load pretrained ResNet50 model for feature extraction
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
base_model.trainable = False  # Freeze the model

# Function to extract CNN features in batches
def extract_cnn_features_batch(images):
    images_preprocessed = preprocess_input(images)
    features = base_model.predict(images_preprocessed, verbose=0)
    return features

# Load data with CNN features
def load_cnn_features(dataset_path, max_samples=100, batch_size=32):
    categories = os.listdir(dataset_path)
    X, y = [], []

    for category in tqdm(categories, desc="Loading categories"):
        cat_path = os.path.join(dataset_path, category)
        batch_images = []
        samples = 0

        for img_name in os.listdir(cat_path)[:max_samples]:
            img_path = os.path.join(cat_path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue

            # Convert grayscale to RGB for ResNet50
            img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            img_resized = cv2.resize(img_rgb, (224, 224))
            batch_images.append(img_resized)
            samples += 1

            # Process batch when full or at the end
            if len(batch_images) == batch_size or (samples == max_samples and batch_images):
                batch_array = np.array(batch_images)
                features = extract_cnn_features_batch(batch_array)
                X.extend(features)
                y.extend([category] * len(features))
                batch_images = []

            if samples >= max_samples:
                break

        # Process remaining images
        if batch_images:
            batch_array = np.array(batch_images)
            features = extract_cnn_features_batch(batch_array)
            X.extend(features)
            y.extend([category] * len(features))

    return np.array(X), np.array(y)

# Load and prepare data
X, y = load_cnn_features(dataset_path, max_samples=100, batch_size=32)
print(f"Loaded {len(X)} samples across {len(np.unique(y))} classes")

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Scale features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train KNN model
knn = KNeighborsClassifier(
    n_neighbors=15,
    weights='distance',
    metric='manhattan',
    algorithm='ball_tree',
    n_jobs=-1
)
knn.fit(X_train_scaled, y_train)

# Evaluate the model
y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.4f}")

Mounted at /content/drive
Categories available: ['frog', 'couch', 'hat', 'book', 'swan', 'computer monitor', 'skull', 'candle', 'camel', 'banana', 'basket', 'wheelbarrow', 'seagull', 'skyscraper', 'calculator', 'loudspeaker', 'paper clip', 'mermaid', 'mouth', 'leaf', 'boomerang', 'stapler', 'purse', 'rooster', 'eye', 'telephone', 'rabbit', 'envelope', 'train', 'parrot', 'bottle opener', 'bathtub', 'parking meter', 'walkie talkie', 'arm', 'snake', 'table', 'guitar', 'eyeglasses', 'piano', 'kangaroo', 'nose', 'grapes', 'teddy-bear', 'rainbow', 'elephant', 'saxophone', 'pipe (for smoking)', 'backpack', 'bulldozer', 'suitcase', 'pumpkin', 'wine-bottle', 't-shirt', 'panda', 'penguin', 'speed-boat', 'parachute', 'fan', 'snail', 'helicopter', 'tablelamp', 'trousers', 'knife', 'hamburger', 'beer-mug', 'armchair', 'cactus', 'spoon', 'shoe', 'giraffe', 'binoculars', 'submarine', 'flashlight', 'chair', 'tooth', 'space shuttle', 'ship', 'race car', 'tree', 'power outlet', 'brain', 'flower with ste

Loading categories: 100%|██████████| 5/5 [01:22<00:00, 16.44s/it]

Loaded 400 samples across 5 classes
Model Accuracy on Test Set: 0.8375


In [6]:
# ## Step 4: Interactive Sketch Drawing Demo

# **Instructions:**
# - Run this cell to open a drawing canvas.
# - Click and drag to draw black lines on a white background.
# - Click "Clear" to reset the canvas.
# - Click "Predict" to see the model’s top-5 predictions.
# - Re-run this cell to draw a new sketch.

from IPython.display import HTML, display, clear_output
import base64
from PIL import Image
import io
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications.resnet50 import preprocess_input
import ipywidgets as widgets

# HTML and JavaScript for the drawing canvas
canvas_html = """
<canvas id="sketchCanvas" width="448" height="448" style="border:1px solid black;"></canvas>
<br>
<button id="clearButton">Clear</button>
<button id="predictButton">Predict</button>
<script>
    var canvas = document.getElementById('sketchCanvas');
    var ctx = canvas.getContext('2d');
    ctx.fillStyle = 'white';
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    ctx.strokeStyle = 'black';
    ctx.lineWidth = 2;
    var drawing = false;

    canvas.addEventListener('mousedown', function(e) {
        drawing = true;
        ctx.beginPath();
        ctx.moveTo(e.offsetX, e.offsetY);
    });

    canvas.addEventListener('mousemove', function(e) {
        if (drawing) {
            ctx.lineTo(e.offsetX, e.offsetY);
            ctx.stroke();
        }
    });

    canvas.addEventListener('mouseup', function() {
        drawing = false;
    });

    canvas.addEventListener('mouseout', function() {
        drawing = false;
    });

    document.getElementById('clearButton').addEventListener('click', function() {
        ctx.fillStyle = 'white';
        ctx.fillRect(0, 0, canvas.width, canvas.height);
    });

    document.getElementById('predictButton').addEventListener('click', function() {
        var dataURL = canvas.toDataURL('image/png');
        google.colab.kernel.invokeFunction('notebook.predict', [dataURL], {});
    });
</script>
"""

# Output widget for predictions
output_widget = widgets.Output()

# Callback function for prediction
def predict_sketch(data_url):
    with output_widget:
        clear_output()

        # Decode the base64 image
        image_data = base64.b64decode(data_url.split(',')[1])
        img = Image.open(io.BytesIO(image_data))
        img = img.convert('RGB')
        img = img.resize((128, 128))  # Match training size
        img_np = np.array(img)

        # Display the sketch
        plt.figure(figsize=(4, 4))
        plt.imshow(img_np)
        plt.title("Your Drawn Sketch")
        plt.axis('off')
        plt.show()

        # Preprocess and predict
        img_preprocessed = preprocess_input(np.expand_dims(img_np, axis=0))
        features = base_model.predict(img_preprocessed, verbose=0)
        features_scaled = scaler.transform(features)
        proba = knn.predict_proba(features_scaled)[0]

        # Top-5 predictions
        top5_indices = np.argsort(proba)[-5:][::-1]
        top5_labels = label_encoder.inverse_transform(top5_indices)
        top5_probs = proba[top5_indices]

        print("Top 5 Predictions:")
        for label, prob in zip(top5_labels, top5_probs):
            print(f"{label}: {prob:.4f}")

# Register the callback with Colab
from google.colab import output
output.register_callback('notebook.predict', predict_sketch)

# Display the canvas and output
display(HTML(canvas_html))
display(output_widget)

Output()

Top 5 Predictions:
swan: 0.4731
frog: 0.3958
hat: 0.0662
book: 0.0649
couch: 0.0000
